# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** c21_surrogate_model_training  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-19
### Properties script
**Goal:** To train a surrogate model on a CSV dataset resulting from structural analyses done in grasshopper, this surrogate model should be able to accurately tell if a beam in the structure is structurally failing or not   
**Inputs:**
*   CSV with structural properties, from grasshopper

**Outputs:**
*   A surrogate model????

# PARAMETERS

In [4]:
import config

CSV_FILE = 'data_3.csv'  # De naam van het CSV-bestand dat je wilt gebruiken (staat in GH_DATA_PATH)

# BENAMING VAN BESTAND EN NETWERK ARCHITECTUUR
schone_naam = CSV_FILE.replace('.csv', '')
delen = schone_naam.split('_')

OPTIMIZATION_TYPE = delen[0] if len(delen) > 0 else 'UNKNOWN'
DATE = delen[1] if len(delen) > 1 else '000000'
ITERATION = delen[2] if len(delen) > 2 else '0000'

prefix_sm = f"{OPTIMIZATION_TYPE}_{DATE}_{ITERATION}"
with open(config.SM_EXPORT_PATH / 'prefix_sm.txt', 'w') as f:
    f.write(prefix_sm)

# IMPORTING DATA

In [9]:
import config
import pandas as pd
import torch
import json

print("1. Dataset inladen...")
df = pd.read_csv(config.GH_DATA_PATH / CSV_FILE)
print(f"✅ Dataset '{CSV_FILE}' ingeladen.")

with open(config.DATA_IO_PATH / 'edge_index.json', 'r') as f:
    edge_index = torch.tensor(json.load(f), dtype=torch.long)

# 2. Harde validatie van de dimensies
print("\n--- DATA VALIDATIE ---")
print(f"Aantal samples (rijen) in dataset: {len(df)}")
print(f"Totaal aantal kolommen in dataset: {df.shape[1]}")
print(f"Dimensie van de edge_index tensor: {edge_index.shape}")

# Korte sanity check
assert df.shape[1] == 72, "Fout: De CSV heeft niet de verwachte 72 kolommen (13 nodes * 3 coords + 32 krachten)!"
assert edge_index.shape[0] == 2, "Fout: edge_index moet precies 2 rijen hebben (source en target nodes)."
assert edge_index.shape[1] == 32, "Fout: edge_index moet precies 32 kolommen (verbindingen) hebben."

print("Validatie succesvol. Data is correct ingeladen.")

1. Dataset inladen...
✅ Dataset 'data_3.csv' ingeladen.

--- DATA VALIDATIE ---
Aantal samples (rijen) in dataset: 10000
Totaal aantal kolommen in dataset: 72
Dimensie van de edge_index tensor: torch.Size([2, 32])
Validatie succesvol. Data is correct ingeladen.


# PROCESSING DATA

In [ ]:
from torch_geometric.data import Data, DataLoader

# 1. Normalisatie (Scaling) instellen
# Neurale netwerken werken wiskundig het best met getallen rond de 0.
# We moeten de X,Y,Z coördinaten en de krachten (kN) dus platdrukken.
print("Start data normalisatie...")
node_scaler = StandardScaler()
edge_scaler = StandardScaler()

# Verzamel alle ruwe data om de scalers te 'trainen' (fitten)
all_node_features = []
all_edge_targets = []

for _, row in df.iterrows():
    for i in range(13):
        all_node_features.append([row[f'v{i}_x'], row[f'v{i}_y'], row[f'v{i}_z']])
    for i in range(32):
        all_edge_targets.append([row[f'beam{i}_a_force']])

node_scaler.fit(all_node_features)
edge_scaler.fit(all_edge_targets)

# 2. Dataset opbouwen met de genormaliseerde data
print("Bouwen van Graph objecten...")
graph_dataset = []

for _, row in df.iterrows():
    # Knooppunten (Nodes)
    nf = [[row[f'v{i}_x'], row[f'v{i}_y'], row[f'v{i}_z']] for i in range(13)]
    nf_scaled = node_scaler.transform(nf)
    x = torch.tensor(nf_scaled, dtype=torch.float)
    
    # Staven (Edges / Targets)
    et = [[row[f'beam{i}_a_force']] for i in range(32)]
    et_scaled = edge_scaler.transform(et)
    y_edge = torch.tensor(et_scaled, dtype=torch.float)
    
    # Combineer x (nodes), edge_index (verbindingen), en y_edge (krachten) tot 1 Graph
    data = Data(x=x, edge_index=edge_index, y_edge=y_edge)
    graph_dataset.append(data)

# 3. Splitsen in Train (80%) en Test (20%)
train_size = int(0.8 * len(graph_dataset))
train_dataset = graph_dataset[:train_size]
test_dataset = graph_dataset[train_size:]

# DataLoaders bepalen hoeveel graphs er tegelijk worden verwerkt (batch_size)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Dataset klaar! Train set: {len(train_dataset)} graphs. Test set: {len(test_dataset)} graphs.")

# MODEL SETUP

In [ ]:
import torch
from torch_geometric.data import Data
import pandas as pd
import numpy as np

def converteer_rij_naar_graaf(rij, edge_index_lijst):
    """
    Zet één rij uit je dataset (1 ontwerp) om in een PyTorch Geometric Data object.
    
    rij: een Pandas Series (één rij uit je dataframe)
    edge_index_lijst: een lijst met verbindingen, bijv. [[0, 0, 1...], [1, 3, 2...]]
    """
    
    # --- 1. KNOOPPUNTEN (Node Features 'x') ---
    # We halen voor v0 t/m v12 de x, y en z coördinaten op.
    node_features = []
    for i in range(13): # Je hebt 13 knooppunten (0 t/m 12)
        x = rij[f'v{i}_x']
        y = rij[f'v{i}_y']
        z = rij[f'v{i}_z']
        node_features.append([x, y, z])
        
    # Converteer de lijst naar een PyTorch Tensor van vorm [13, 3]
    tensor_x = torch.tensor(node_features, dtype=torch.float)
    
    # --- 2. TOPOLOGIE (Edge Index) ---
    # De edge index moet een tensor zijn van vorm [2, 32] met gehele getallen (long)
    tensor_edge_index = torch.tensor(edge_index_lijst, dtype=torch.long)
    
    # --- 3. DOELWAARDES (Targets 'y') ---
    # We halen de utilization op voor beam0 t/m beam31
    targets = []
    for i in range(32): # Je hebt 32 balken
        utilization = rij[f'beam{i}_utilization']
        targets.append([utilization]) # Let op de extra haakjes, we willen vorm [32, 1]
        
    tensor_y = torch.tensor(targets, dtype=torch.float)
    
    # --- 4. MAAK HET GNN OBJECT ---
    # We stoppen alles samen in één 'Data' object dat het GNN snapt
    graaf_data = Data(x=tensor_x, edge_index=tensor_edge_index, y=tensor_y)
    
    return graaf_data

print("✅ Data-converter functie succesvol geladen!")

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

# --- 1. HET GNN MODEL DEFINIËREN ---
class ConstructieGNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Eerste laag: we stoppen er 3 waardes in (X, Y, Z) 
        # en maken er een rijkere representatie van (16 waardes)
        self.conv1 = GCNConv(in_channels=3, out_channels=16)
        
        # Tweede laag: we filteren de informatie nog verder
        self.conv2 = GCNConv(in_channels=16, out_channels=16)
        
        # Laatste stap: we voorspellen de 'utilization' (1 waarde per balk)
        self.balk_voorspeller = torch.nn.Linear(in_features=16, out_features=1)

    def forward(self, data):
        # Haal de coördinaten (x) en de structuur (edge_index) op
        x, edge_index = data.x, data.edge_index

        # --- Message Passing (Knooppunten praten met elkaar) ---
        x = self.conv1(x, edge_index)
        x = F.relu(x) # Een wiskundige filter (activeert de neuronen)
        x = self.conv2(x, edge_index)

        # --- Balk Belasting Voorspellen ---
        # Om de belasting van een balk te voorspellen, pakken we de informatie 
        # van de twee knooppunten die aan weerszijden van de balk zitten.
        knooppunt_A = x[edge_index[0]] # Beginpunten van de balken
        knooppunt_B = x[edge_index[1]] # Eindpunten van de balken
        
        # We combineren de data van de twee knooppunten (bijv. door ze op te tellen)
        balk_informatie = knooppunt_A + knooppunt_B
        
        # We voorspellen de belasting (utilization)
        utilization_voorspelling = self.balk_voorspeller(balk_informatie)
        
        return utilization_voorspelling

# (In de praktijk zouden we hier jouw dataset inladen en het model trainen)
print("Het GNN-model is succesvol aangemaakt!")

In [ ]:
# We maken een lege lijst waar alle omgezette vakwerken in komen
gnn_dataset = []

# Loop door elke rij (elk ontwerp) in je dataframe
for index, rij in df.iterrows():
    graaf = converteer_rij_naar_graaf(rij, edge_index)
    gnn_dataset.append(graaf)

print(f"Klaar! Er zijn {len(gnn_dataset)} vakwerken succesvol omgezet voor het GNN.")
# Laten we even spieken naar het eerste vakwerk:
print(gnn_dataset[0])

# MODEL TRAINING

In [5]:
import random
from torch_geometric.loader import DataLoader

# 1. Hussel de data (zodat het model niet per ongeluk patronen in de volgorde leert)
random.seed(42) # Dit zorgt ervoor dat de 'willekeur' elke keer hetzelfde is, handig voor je onderzoek!
random.shuffle(gnn_dataset)

# 2. Bepaal de verdeling (80% trainen, 20% testen)
train_grootte = int(0.8 * len(gnn_dataset))
train_dataset = gnn_dataset[:train_grootte]
test_dataset = gnn_dataset[train_grootte:]

# 3. Maak DataLoaders (deze maken pakketjes van bijv. 64 vakwerken tegelijk)
# shuffle=True bij training zorgt dat de pakketjes elke 'ronde' (epoch) anders zijn
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Dataset succesvol gesplitst!")
print(f"Aantal constructies om van te leren (Train): {len(train_dataset)}")
print(f"Aantal constructies om mee te testen (Test): {len(test_dataset)}")

Dataset ingeladen met 10000 samples.
Data gesplitst: 8000 trainingsamples en 2000 testsamples.
3. Data normaliseren...

Voorbeeld van de genormaliseerde Training Data (X):
   v0_x  v0_y  v0_z      v1_x  v1_y  v1_z  v2_x  v2_y  v2_z  v3_x  ...  \
0   0.0   0.0   0.0  0.666667   0.0   0.0   0.0   0.0   0.0   0.0  ...   
1   0.0   0.0   0.0  1.000000   0.0   0.0   0.0   0.0   0.0   0.0  ...   
2   0.0   0.0   0.0  0.500000   0.0   0.0   0.0   0.0   0.0   0.0  ...   

   beam21_utilization  beam22_utilization  beam23_utilization  \
0            0.320048            0.120944            0.051751   
1            0.456580            0.005617            0.015198   
2            0.372657            0.112954            0.048886   

   beam24_utilization  beam25_utilization  beam26_utilization  \
0            0.121946            0.000873            0.147322   
1            0.175351            0.007852            0.158951   
2            0.064241            0.007308            0.153010   

   beam27

In [28]:
import torch.nn as nn
import torch.optim as optim

# 1. Instellingen voor het leerproces
model = ConstructieGNN()
loss_functie = nn.MSELoss() # Meet het kwadratische verschil tussen voorspelling en realiteit
optimizer = optim.Adam(model.parameters(), lr=0.005) # lr is de 'learning rate' (stapgrootte)

aantal_epochs = 50 # We laten het model 50 keer door de hele dataset gaan

# 2. De Trainingslus
print("Start met trainen van het GNN...\n")

for epoch in range(aantal_epochs):
    model.train() # Zet het model in de trainingsmodus
    totale_loss = 0

    for batch in train_loader:
        optimizer.zero_grad() # Reset de oude berekeningen van de vorige stap
        
        # 1. Laat het model een voorspelling doen voor deze batch constructies
        voorspellingen = model(batch)
        
        # 2. Bereken de foutmarge vergeleken met de werkelijke data (batch.y)
        loss = loss_functie(voorspellingen, batch.y)
        
        # 3. Backpropagation: leer van de fouten en pas het netwerk aan
        loss.backward()
        optimizer.step()
        
        totale_loss += loss.item()

    # Print de voortgang elke 10 epochs
    gemiddelde_loss = totale_loss / len(train_loader)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:02d}/{aantal_epochs} | Gemiddelde Fout (Loss): {gemiddelde_loss:.6f}')

print("\nTraining succesvol voltooid!")

Start met trainen van het GNN...

Epoch 01/50 | Gemiddelde Fout (Loss): 0.089905
Epoch 10/50 | Gemiddelde Fout (Loss): 0.024126
Epoch 20/50 | Gemiddelde Fout (Loss): 0.023900
Epoch 30/50 | Gemiddelde Fout (Loss): 0.023418
Epoch 40/50 | Gemiddelde Fout (Loss): 0.023296
Epoch 50/50 | Gemiddelde Fout (Loss): 0.023238

Training succesvol voltooid!


# MODEL TESTING

In [6]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# 1. Verzamel alle voorspellingen en echte waardes
model.eval()
echte_waardes = []
voorspelde_waardes = []

with torch.no_grad():
    for batch in test_loader:
        voorspellingen = model(batch)
        
        # Converteer de PyTorch tensors naar platte Numpy lijsten
        echte_waardes.extend(batch.y.cpu().numpy().flatten())
        voorspelde_waardes.extend(voorspellingen.cpu().numpy().flatten())

echte_waardes = np.array(echte_waardes)
voorspelde_waardes = np.array(voorspelde_waardes)

# 2. Maak de figuren aan
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- GRAFIEK 1: Predicted vs Actual Scatterplot ---
ax1.scatter(echte_waardes, voorspelde_waardes, alpha=0.3, color='#1f77b4', s=10, edgecolor='none')

# Bereken de min en max voor de ideale rode stippellijn (y = x)
min_val = min(echte_waardes.min(), voorspelde_waardes.min())
max_val = max(echte_waardes.max(), voorspelde_waardes.max())
ax1.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label='Ideale Voorspelling (y=x)')

ax1.set_title('GNN Voorspelling vs. Werkelijke Utilization', fontsize=12, fontweight='bold')
ax1.set_xlabel('Werkelijke Utilization (FEA Solver)', fontsize=11)
ax1.set_ylabel('Voorspelde Utilization (GNN)', fontsize=11)
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)

# --- GRAFIEK 2: Foutmarge Verdeling (Histogram) ---
fouten = voorspelde_waardes - echte_waardes
ax2.hist(fouten, bins=50, color='#ff7f0e', alpha=0.7, edgecolor='black', linewidth=0.5)

ax2.axvline(0, color='red', linestyle='--', linewidth=2, label='Nul Foutmarge')
ax2.set_title('Verdeling van de Voorspellingsfouten (Residuals)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Afwijking (Voorspeld - Werkelijk)', fontsize=11)
ax2.set_ylabel('Frequentie', fontsize=11)
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


⚙️ Neuraal Netwerk aan het trainen...


c:\Users\jaspe\Documents\PyEnvs\thesis_env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:1775: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


✅ Training voltooid!

📊 --- RESULTATEN VAN HET MODEL ---
R2 Score (Determinatiecoëfficiënt): 0.9164 (1.0 is perfect)
Mean Squared Error (Foutmarge):   0.0007 (0.0 is perfect)

🔍 --- VOORBEELD ITERATIE (Index 5) ---
Werkelijke beam31_utilization: 	0.057
AI Voorspelt beam31_utilization: 	0.057
----------------------------------------


# EXPORT

In [ ]:
# We slaan de meetlatten op als .pkl bestanden.
# Deze heb je later nodig om de voorspellingen van je AI weer om te rekenen naar echte millimeters en kilo's!
joblib.dump(scaler_X, config.SM_EXPORT_PATH / f'scaler_X_.pkl')
joblib.dump(scaler_y, config.SM_EXPORT_PATH / f'scaler_y_{prefix_sm}.pkl')

print(f"\n✅ Scalers succesvol opgeslagen onder {config.SM_EXPORT_PATH}")

# Exporteer het getrainde netwerk zodat we het later in de optimalisatie-loop kunnen inladen!
joblib.dump(surrogate_model, config.SM_EXPORT_PATH / f'surrogate_model_{prefix_sm}.pkl')

print(f"\n💾 Model succesvol opgeslagen als onder {config.SM_EXPORT_PATH}")